In [1]:
import sys
!{sys.executable} -m pip install transformers torch            

In [2]:
from transformers import AutoTokenizer

In [3]:
import torch
import torch.nn.functional as F

In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "bert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
data = [
    # बैंक
    ("मैं बैंक गया", 0),
    ("उसने बैंक में पैसा जमा किया", 0),
    ("नदी का बैंक सुंदर है", 1),
    ("हम बैंक के किनारे बैठे", 1),

    
    # कल
    ("मैं कल आया", 0),
    ("वह कल स्कूल गया था", 0),
    ("मैं कल जाऊंगा", 1),
    ("वह कल परीक्षा देगा", 1),

    # फल
    ("फल खाना सेहत के लिए अच्छा है", 0),
    ("पेड़ पर फल लगे हैं", 0),
    ("मेहनत का फल मीठा होता है", 1),
    ("उसे मेहनत का फल मिला", 1),

    # हाथ
    ("उसका हाथ टूट गया", 0),
    ("मेरे हाथ ठंडे हैं", 0),
    ("उसने मेरी मदद के लिए हाथ बढ़ाया", 1),
    ("हमें एक दूसरे का हाथ पकड़ना चाहिए", 1),

    # सिर
    ("उसके सिर में दर्द है", 0),
    ("सिर पर चोट लगी", 0),
    ("वह कंपनी का सिर है", 1),d
    ("पहाड़ का सिर बर्फ से ढका है", 1),
]

In [6]:
texts = [x[0] for x in data]
labels = torch.tensor([x[1] for x in data])

encodings = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")

In [7]:
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)

model.train()
for epoch in range(3):
    outputs = model(**encodings, labels=labels)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    print(f"Epoch {epoch+1}, Loss:", loss.item())

print("Training Done ✅")

Epoch 1, Loss: 0.6892687678337097
Epoch 2, Loss: 0.6403631567955017
Epoch 3, Loss: 0.5393059253692627
Training Done ✅


In [8]:
def predict(sentence):
    model.eval()
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    probs = F.softmax(outputs.logits, dim=1)
    pred = torch.argmax(probs).item()

    # Meaning mapping
    if "बैंक" in sentence:
        if pred == 0:
            meaning = "Financial Bank"
        else:
            meaning = "River Bank"

    elif "कल" in sentence:
        if pred == 0:
            meaning = "Past (Yesterday)"
        else:
            meaning = "Future (Tomorrow)"

    elif "फल" in sentence:
        if pred == 0:
            meaning = "Fruit"
        else:
            meaning = "Result"

    elif "हाथ" in sentence:
        if pred == 0:
            meaning = "Body Part (Hand)"
        else:
            meaning = "Help/Support"

    elif "सिर" in sentence:
        if pred == 0:
            meaning = "Head (Body)"
        else:
            meaning = "Top/Leader"

    else:
        meaning = "Unknown"

    print(f"\nSentence: {sentence}")
    print("Predicted Meaning:", meaning)
    print("Confidence:", probs.detach().numpy())

In [ ]:
while True:
    sentence = input("\nEnter a Hindi sentence (type 'exit' to stop): ")
    
    if sentence.lower() == "exit":
        break

    # Safety check
    if not any(word in sentence for word in ["बैंक", "कल", "फल", "हाथ", "सिर"]):
        print(" No known ambiguous word found.")
    else:
        predict(sentence)


Enter a Hindi sentence (type 'exit' to stop):  उसने मेरी मदद के लिए हाथ बढ़ाया



Sentence: उसने मेरी मदद के लिए हाथ बढ़ाया
Predicted Meaning: Help/Support
Confidence: [[0.29283252 0.70716745]]



Enter a Hindi sentence (type 'exit' to stop):  उसका हाथ टूट गया



Sentence: उसका हाथ टूट गया
Predicted Meaning: Body Part (Hand)
Confidence: [[0.7388267  0.26117334]]
